# GeoLift Testing Pipeline - Example Usage

This notebook demonstrates how to use the GeoLift pipeline for geo experiment analysis.

## 1. Setup

First, let's import the necessary libraries and generate sample data.

In [ ]:
from geolift import GeoLift
from generate_sample_data import generate_sample_data
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## 2. Generate Sample Data

Generate synthetic geo experiment data with a known treatment effect.

In [ ]:
# Generate sample data
df = generate_sample_data(
    n_locations=10,
    n_days=180,
    treatment_locations=['chicago', 'houston'],
    treatment_start_day=120,
    treatment_lift=0.15,  # 15% lift
    output_file='sample_data.csv'
)

print("\nSample of generated data:")
df.head(20)

## 3. Initialize GeoLift and Load Data

In [ ]:
# Initialize GeoLift
gl = GeoLift()

# Load data
gl.load_data('sample_data.csv')

## 4. Data Exploration

In [ ]:
# Show summary statistics
summary = gl.summary_stats()

In [ ]:
# Visualize trends
gl.plot_trends()

## 5. Pre-Test Analysis

Before running the test, analyze the fit between treatment and control locations.

In [ ]:
# Analyze pre-treatment fit
treatment_locs = ['chicago', 'houston']
fit_results = gl.analyze_fit(
    treatment_locations=treatment_locs,
    pre_period_end='2024-04-30'  # Before treatment starts
)

## 6. Power Analysis

Estimate the Minimum Detectable Effect (MDE) for different test durations.

In [ ]:
# Estimate MDE for 60-day test
mde_results = gl.estimate_mde(
    treatment_locations=treatment_locs,
    test_duration_days=60,
    alpha=0.05,
    power=0.8,
    pre_period_end='2024-04-30'
)

In [ ]:
# Compare MDEs for different test durations
import matplotlib.pyplot as plt

durations = [14, 30, 45, 60, 90]
mde_values = []

for duration in durations:
    result = gl.estimate_mde(
        treatment_locations=treatment_locs,
        test_duration_days=duration,
        pre_period_end='2024-04-30'
    )
    mde_values.append(result['mde_relative_pct'])

plt.figure(figsize=(10, 6))
plt.plot(durations, mde_values, 'o-', linewidth=2, markersize=8)
plt.xlabel('Test Duration (days)', fontsize=12)
plt.ylabel('MDE (%)', fontsize=12)
plt.title('Minimum Detectable Effect vs Test Duration', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Run Synthetic Control Test

Run the geo experiment analysis using synthetic control methodology.

In [ ]:
# Run test
results = gl.run_test(
    treatment_locations=['chicago', 'houston'],
    treatment_start='2024-05-01',
    treatment_end='2024-06-29'
)

## 8. Visualize Results

In [ ]:
# Plot results
gl.plot_results()

## 9. Summary Statistics

In [ ]:
# Print summary
summary_results = gl.summary()

In [ ]:
# Access specific results
print(f"Cumulative Lift: ${summary_results['cumulative_effect']:,.2f}")
print(f"Relative Effect: {summary_results['relative_effect_pct']:.2f}%")
print(f"P-value: {summary_results['p_value']:.4f}")
print(f"Statistically Significant: {summary_results['significant']}")

## 10. Export Results

In [ ]:
# Export results to CSV
gl.export_results('geolift_results.csv')

# Preview exported results
results_df = pd.read_csv('geolift_results.csv')
print("\nExported results (first 10 rows):")
results_df.head(10)

## 11. Using Your Own Data

To use your own data, simply prepare a CSV with the required columns and follow the same workflow:

In [ ]:
# Example with custom data
# gl_custom = GeoLift()
# gl_custom.load_data('your_data.csv', 
#                     date_col='date',
#                     location_col='city',
#                     metric_col='conversions',
#                     spend_col='media_spend')
# 
# gl_custom.summary_stats()
# gl_custom.plot_trends()
# 
# gl_custom.run_test(
#     treatment_locations=['your_city_1', 'your_city_2'],
#     treatment_start='2024-01-15',
#     treatment_end='2024-02-15'
# )
# 
# gl_custom.plot_results()
# gl_custom.summary()

## 12. Comparing with Haus Results

To validate against Haus:

1. **Input the same data**: Use the same CSV for both platforms
2. **Match parameters**: Use the same treatment locations, dates, and control groups
3. **Compare metrics**:
   - Cumulative lift estimate
   - Confidence intervals
   - P-values
   - Relative effect percentages
4. **Check methodology**: Both should use synthetic control/ASCM approaches

The GeoLift pipeline uses standard synthetic control methodology that should align closely with Haus outputs.